# Text, Event, and FinBERT Pipeline

这个 notebook 串联 Stage 1 的核心文本处理流程：原始数据清洗、候选事件生成、FinBERT 推理，以及事件与模型输出的标准化。

建议按顺序执行各部分；如果已有中间产物，也可以从对应章节开始继续。

## Part 1. Raw Data Cleaning

统一社交媒体原始数据的 schema、时间字段和主表结构。

# Cell 0: 环境检测与依赖安装
## 检查必要依赖是否已安装，若缺失则自动安装
**依赖清单**：pandas, numpy, python-dateutil, pytz, pandas_market_calendars, ftfy
**目的**：确保清洗脚本可在一个新环境中独立运行

In [1]:
import subprocess
import sys

# 需要检测的包列表
REQUIRED_PACKAGES = [
    "pandas",
    "numpy",
    "python-dateutil",
    "pytz",
    "pandas_market_calendars",
    "ftfy"
]

def check_and_install(package):
    """检查单个包，未安装则尝试安装"""
    try:
        __import__(package)
        print(f"[OK] {package} 已安装")
    except ImportError:
        print(f"[INSTALLING] {package} 未找到，正在安装...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])
        print(f"[OK] {package} 安装完成")

for pkg in REQUIRED_PACKAGES:
    check_and_install(pkg)

print("\n所有依赖检测完毕。")

[OK] pandas 已安装
[OK] numpy 已安装
[INSTALLING] python-dateutil 未找到，正在安装...
[OK] python-dateutil 安装完成
[OK] pytz 已安装
[OK] pandas_market_calendars 已安装
[OK] ftfy 已安装

所有依赖检测完毕。



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Cell 1: 路径配置 + 依赖导入
## 定义原始数据路径、输出路径，以及全局常量（日历边界、term_id 边界等）
**日历覆盖范围**：2017-01-20（第一任期开始）至 2025-10-25（Truth Social 数据尾端）
**term_id 边界**：
- first_term：2017-01-20 至 2021-01-08
- second_term：2025-01-20 至 2025-10-25
- out_of_term：其余时间

In [2]:
import os
import hashlib
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import pytz
from datetime import datetime, timedelta
from dateutil import parser as date_parser

import pandas_market_calendars as mcal

# ============================================================
# 1. 路径配置
# ============================================================
PROJECT_ROOT = Path("/Users/horace/Coding/ML finance/project")
DATA_RAW    = PROJECT_ROOT / "data"
DATA_OUT    = PROJECT_ROOT / "data" / "processed"
DATA_OUT.mkdir(parents=True, exist_ok=True)

TWEETS_CSV     = DATA_RAW / "tweets.csv"          # Twitter 原始数据
TRUTH_CSV      = DATA_RAW / "truth_archive.csv"   # Truth Social 原始数据

# ============================================================
# 2. 全局常量：term_id 边界（按就职时点定义）
# ============================================================
# 就职按美东 12:00 计算，统一换算为 UTC，避免把就职前内容误归入 in-term
FIRST_TERM_START  = pd.Timestamp("2017-01-20 17:00:00", tz="UTC")
FIRST_TERM_END    = pd.Timestamp("2021-01-08 23:59:59", tz="UTC")
SECOND_TERM_START = pd.Timestamp("2025-01-20 17:00:00", tz="UTC")
SECOND_TERM_END   = pd.Timestamp("2025-10-25 23:59:59", tz="UTC")

# NYSE 交易日历覆盖范围（首尾各留 5 年缓冲）
CAL_START = "2008-01-01"
CAL_END   = "2026-12-31"

warnings.filterwarnings("ignore")
print(f"项目根目录：{PROJECT_ROOT}")
print(f"输出目录：{DATA_OUT}")
print(f"term_id 边界已设置：first=[{FIRST_TERM_START}, {FIRST_TERM_END}] "
      f"second=[{SECOND_TERM_START}, {SECOND_TERM_END}]")

项目根目录：/Users/horace/Coding/ML finance/project
输出目录：/Users/horace/Coding/ML finance/project/data/processed
term_id 边界已设置：first=[2017-01-20 17:00:00+00:00, 2021-01-08 23:59:59+00:00] second=[2025-01-20 17:00:00+00:00, 2025-10-25 23:59:59+00:00]


# Cell 2: 读取原始 CSV（初览）
## 读取 tweets.csv（Twitter）和 truth_archive.csv（Truth Social），打印 shape、dtype、缺失率、样本行
**目的**：验证文件读取正常，发现原始数据中可能存在的问题（缺失列、异常格式等）

In [3]:
# ============================================================
# 读取 Twitter 数据（tweets.csv）
# 字段：id, text, isRetweet, isDeleted, device, favorites, retweets, date, isFlagged
# ============================================================
df_twitter = pd.read_csv(TWEETS_CSV, low_memory=False)

print("=" * 60)
print("tweets.csv（Twitter）")
print("=" * 60)
print(f"Shape：{df_twitter.shape}")
print(f"\n列类型：\n{df_twitter.dtypes}")
print(f"\n缺失率：\n{df_twitter.isnull().mean().round(4) * 100}%")
print(f"\n前 5 行预览：\n{df_twitter.head()}")

# ============================================================
# 读取 Truth Social 数据（truth_archive.csv）
# 字段：id, created_at, content, url, media, replies_count, reblogs_count, favourites_count
# ============================================================
df_truth = pd.read_csv(TRUTH_CSV, low_memory=False)

print("\n" + "=" * 60)
print("truth_archive.csv（Truth Social）")
print("=" * 60)
print(f"Shape：{df_truth.shape}")
print(f"\n列类型：\n{df_truth.dtypes}")
print(f"\n缺失率：\n{df_truth.isnull().mean().round(4) * 100}%")
print(f"\n前 5 行预览：\n{df_truth.head()}")

tweets.csv（Twitter）
Shape：(56571, 9)

列类型：
id           int64
text           str
isRetweet      str
isDeleted      str
device         str
favorites    int64
retweets     int64
date           str
isFlagged      str
dtype: object

缺失率：
id           0.0
text         0.0
isRetweet    0.0
isDeleted    0.0
device       0.0
favorites    0.0
retweets     0.0
date         0.0
isFlagged    0.0
dtype: float64%

前 5 行预览：
                    id                                               text  \
0    98454970654916608  Republicans and Democrats have both created ou...   
1  1234653427789070336  I was thrilled to be back in the Great city of...   
2  1218010753434820614  RT @CBS_Herridge: READ: Letter to surveillance...   
3  1304875170860015617  The Unsolicited Mail In Ballot Scam is a major...   
4  1218159531554897920  RT @MZHemingway: Very friendly telling of even...   

  isRetweet isDeleted              device  favorites  retweets  \
0         f         f           TweetDeck         49      

# Cell 3: Schema 统一映射 + Truth Social 转发检测（升级版）
## 将两个数据源映射到统一字段体系：
统一字段：`text_id`, `source`, `published_at_utc`, `text_raw`, `is_retweet`, `device`, `term_id`

### Truth Social 转发检测逻辑（优先级递减）：
1. 检查是否有独立字段 `is_reblog` / `is_retruth`（布尔值）
2. 检查文本前 150 字符是否含 `RT @`（Twitter 风格转发）
3. 检查文本前 150 字符是否以 `ReTruthed` 开头（Truth 风格转发）
4. 若命中任意一条 → `is_retweet = True`

In [4]:
# ============================================================
# Part A：映射 tweets.csv（Twitter）
# ============================================================
df_tw = df_twitter.rename(columns={
    "id":        "text_id",
    "text":      "text_raw",
    "date":      "published_at_utc",
    "isRetweet": "is_retweet_flag"   # t / f 字符串
})[["text_id", "text_raw", "published_at_utc", "is_retweet_flag", "device"]].copy()

# 布尔化 is_retweet（t → True，f → False）
df_tw["is_retweet"] = df_tw["is_retweet_flag"].apply(lambda x: str(x).lower() == "t")
df_tw["is_trump_direct_text"] = ~df_tw["is_retweet"]
df_tw["source"] = "twitter"
df_tw.drop(columns=["is_retweet_flag"], inplace=True)

# 类型转换：published_at_utc → datetime
df_tw["published_at_utc"] = pd.to_datetime(df_tw["published_at_utc"], utc=True, errors="coerce")

print("[Twitter] Schema 映射完成")
print(f"  text_id 唯一数：{df_tw['text_id'].nunique()} / 总行数：{len(df_tw)}")
print(f"  is_retweet=True 占比：{df_tw['is_retweet'].mean():.2%}")

# ============================================================
# Part B：映射 truth_archive.csv（Truth Social）
# ============================================================
df_ts = df_truth.rename(columns={
    "id":        "text_id",
    "content":   "text_raw",
    "created_at":"published_at_utc"
})[["text_id", "text_raw", "published_at_utc", "url"]].copy()

df_ts["source"] = "truthsocial"
df_ts["device"] = None   # Truth Social 数据中无 device 字段，设为 None

# 类型转换：published_at_utc → datetime（ISO8601 UTC）
df_ts["published_at_utc"] = pd.to_datetime(df_ts["published_at_utc"], utc=True, errors="coerce")

# ============================================================
# Part C：Truth Social 转发检测（升级版：三重检测）
# ============================================================
def detect_retweet_truthsocial(row):
    """
    优先级：
    1. 独立字段 is_reblog / is_retruth
    2. 文本前 150 字符含 RT @（Twitter 风格）
    3. 文本前 150 字符以 ReTruthed 开头（Truth 风格）
    """
    text = str(row.get("text_raw", ""))[:150]

    # 检查 1：独立布尔字段（如果存在的话）
    # （truth_archive.csv 当前无此字段，保留扩展性）
    # if row.get("is_reblog") is True or row.get("is_retruth") is True:
    #     return True

    # 检查 2：RT @ 前缀
    if "RT @" in text:
        return True

    # 检查 3：ReTruthed 开头（大小写敏感）
    if text.startswith("ReTruthed"):
        return True

    return False

df_ts["is_retweet"] = df_ts.apply(detect_retweet_truthsocial, axis=1)
df_ts["is_trump_direct_text"] = ~df_ts["is_retweet"]

print("\n[Truth Social] Schema 映射完成")
print(f"  text_id 唯一数：{df_ts['text_id'].nunique()} / 总行数：{len(df_ts)}")
print(f"  is_retweet=True 占比：{df_ts['is_retweet'].mean():.2%}")

# 预览检测到的转发样本
rt_samples = df_ts[df_ts["is_retweet"]]["text_raw"].head(3).tolist()
print(f"\nTruth Social 转发样本（前 3 条）：")
for i, s in enumerate(rt_samples, 1):
    print(f"  {i}. {s[:100]}...")

[Twitter] Schema 映射完成
  text_id 唯一数：56571 / 总行数：56571
  is_retweet=True 占比：17.46%

[Truth Social] Schema 映射完成
  text_id 唯一数：29409 / 总行数：29409
  is_retweet=True 占比：17.74%

Truth Social 转发样本（前 3 条）：
  1. RT @realDonaldTrumpCanada was caught, red handed, putting up a fraudulent advertisement on Ronald Re...
  2. RT @realDonaldTrumpA âBlue Slipâ means that if youâre a Republican President, and there happen...
  3. RT @realDonaldTrumpThe meeting with President Volodymyr Zelenskyy of Ukraine was very interesting, a...


# Cell 4: 文本质量修复（FinBERT 感知，最小化语义破坏）
## 清洗规则（不破坏情绪信号）：
- **HTML 实体解码**：`&amp;` → `&`；`&lt;` → `<`；`&gt;` → `>`；`&quot;` → `"`
- **mojibake 修复**：用 `ftfy` 自动修复编码异常（如 `â¢` → `'`）
- **异常替换字符**：将 `�`、控制字符等替换为空格
- **URL 占位符**：原始 URL 在 `url` 字段中物理保留；`text_clean` 中用 `[URL]` 统一替换，避免 FinBERT 词表噪音
- **空白符归一化**：多个连续换行符/空格 → 单个空格，提高 FinBERT 上下文捕捉质量
- **不删除标点符号**：感叹号（!）、问号（?）、全大写词汇全部保留
- **大小写敏感**：不强制 lowercase（FinBERT 对大小写敏感）

**FinBERT 感知说明**：FinBERT 会学习到大写、感叹号、标点的情绪含义，清洗时保留这些特征。`[URL]` 占位符保留了"此处有外部链接"的信号，但消除了具体链接对语义空间的干扰。

In [5]:
import ftfy
import re
import html

# ============================================================
# 文本清洗函数（FinBERT 感知）
# ============================================================
# URL 正则：匹配 https?:// 开头后跟非空白字符的完整 URL
URL_PATTERN = re.compile(r"https?://\S+", re.IGNORECASE)

def clean_text_finbert_aware(text):
    """
    最小化文本清洗，保留情绪信号（感叹号、全大写、标点）。
    处理步骤（严格按顺序）：
    1. None / NaN → 空字符串
    2. HTML 实体解码
    3. ftfy mojibake 修复
    4. URL → [URL] 占位符（避免 FinBERT 词表噪音；原始 URL 在 url 字段物理保留）
    5. 异常替换字符（控制字符、�）→ 空格
    6. 空白符归一化：多个连续换行符/空格 → 单个空格
    7. 前后空白strip
    """
    if pd.isna(text) or text is None:
        return ""

    text = str(text)

    # Step 1：HTML 实体解码
    text = html.unescape(text)

    # Step 2：ftfy mojibake 修复
    # ftfy 可自动检测并修复常见编码错误（如 UTF-8 被误读为 Latin-1）
    text = ftfy.fix_text(text)

    # Step 3：URL → [URL] 占位符
    # 原始 URL 在 url 字段物理保留；此处统一替换，消除噪音同时保留"有链接"信号
    text = URL_PATTERN.sub("[URL]", text)

    # Step 4：异常替换字符 → 空格
    # 保留 ASCII 可打印字符 + 常见标点，不删 ! ? 全大写
    # 控制字符（C0/C1）和 replacement character → 替换为空格
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\uFFFD]", " ", text)

    # Step 5：空白符归一化（多个连续换行/空格 → 单个空格）
    # 此步同时处理 Twitter/Truth Social 中常见的冗余回车
    text = re.sub(r"\s+", " ", text)

    # Step 6：前后空白strip
    text = text.strip()

    return text


# ============================================================
# 修复两个 DataFrame 的文本
# ============================================================
print("开始文本清洗...")

df_tw["text_clean"] = df_tw["text_raw"].apply(clean_text_finbert_aware)
df_ts["text_clean"] = df_ts["text_raw"].apply(clean_text_finbert_aware)

# ============================================================
# 质量报告：清洗前后对比
# ============================================================
def text_quality_report(df, source_name):
    """输出文本质量统计"""
    total = len(df)
    empty = (df["text_clean"] == "").sum()
    same_as_raw = (df["text_clean"] == df["text_raw"].astype(str)).sum()
    has_exclamation = df["text_clean"].str.contains("!", regex=False).sum()
    has_allcaps = df["text_clean"].apply(
        lambda x: any(w.isupper() and len(w) > 2 for w in str(x).split())
    ).sum()
    url_placeholder_count = df["text_clean"].str.contains(r"\[URL\]", regex=True, na=False).sum()

    print(f"\n[{source_name}] 文本质量报告（{total} 条）")
    print(f"  空文本（清洗后）：{empty} ({empty/total:.2%})")
    print(f"  与原文完全相同：{same_as_raw} ({same_as_raw/total:.2%})")
    print(f"  含感叹号 !：{has_exclamation} ({has_exclamation/total:.2%})")
    print(f"  含全大写词：{has_allcaps} ({has_allcaps/total:.2%})")
    print(f"  含 [URL] 占位符：{url_placeholder_count} ({url_placeholder_count/total:.2%})")

text_quality_report(df_tw, "Twitter")
text_quality_report(df_ts, "Truth Social")

# 预览清洗前后对比（取含 URL 的样本）
print("\n清洗前后对比样本（含 URL 的前 3 条）：")
sample_idx = df_tw[df_tw["text_clean"].str.contains(r"\[URL\]", regex=True, na=False)].head(3).index
for idx in sample_idx:
    print(f"\n[清洗前] {df_tw.loc[idx, 'text_raw'][:150]}")
    print(f"[清洗后] {df_tw.loc[idx, 'text_clean'][:150]}")

开始文本清洗...

[Twitter] 文本质量报告（56571 条）
  空文本（清洗后）：0 (0.00%)
  与原文完全相同：24945 (44.10%)
  含感叹号 !：26015 (45.99%)
  含全大写词：28928 (51.14%)
  含 [URL] 占位符：15892 (28.09%)

[Truth Social] 文本质量报告（29409 条）
  空文本（清洗后）：5053 (17.18%)
  与原文完全相同：6327 (21.51%)
  含感叹号 !：11148 (37.91%)
  含全大写词：18691 (63.56%)
  含 [URL] 占位符：10702 (36.39%)

清洗前后对比样本（含 URL 的前 3 条）：

[清洗前] I was thrilled to be back in the Great city of Charlotte, North Carolina with thousands of hardworking American Patriots who love our Country, cherish
[清洗后] I was thrilled to be back in the Great city of Charlotte, North Carolina with thousands of hardworking American Patriots who love our Country, cherish

[清洗前] RT @WhiteHouse: President @realDonaldTrump announced historic steps to protect the Constitutional right to pray in public schools! https://…
[清洗后] RT @WhiteHouse: President @realDonaldTrump announced historic steps to protect the Constitutional right to pray in public schools! [URL]

[清洗前] Getting a little exercise this morning! https:

# Cell 5: 时间解析 + 交易日映射（时区处理顺序：UTC → US/Eastern → 交易时段判定）
## 时区处理三步骤（严格按此顺序，DST 自动处理）：
1. **第一步**：统一转为 UTC
2. **第二步**：映射到 `US/Eastern`（pytz / pandas 时区转换，DST 自动处理）
3. **第三步**：判定是否处于 09:30–16:00 ET 交易时段

## market_effective_date 映射规则：
| 发布时刻 | market_effective_date |
|---|---|
| 美东交易时段 09:30–16:00 内 | 当日（T） |
| 盘后（16:00 后）、开盘前（09:30 前）、周末、休市 | 下一交易日（T+1） |

## 生成交易日骨架表（calendar spine）：
使用 `pandas_market_calendars` 获取 NYSE 交易日序列，作为所有数据的主键和对齐基准。

In [6]:
# ============================================================
# Step 1 & 2：时间解析（确保 UTC）
# ============================================================
# tweets.csv 的 date 字段：假设为本地时间（非 UTC），需指定时区
# Twitter 发帖时间通常以服务器本地时间存储，这里先尝试解析再统一为 UTC

def parse_twitter_time(ts_str):
    """
    解析 Twitter 日期字符串，先推断时区再转 UTC。
    Twitter API v1.5 返回的时间通常是 UTC。
    """
    if pd.isna(ts_str):
        return pd.NaT
    try:
        dt = date_parser.parse(str(ts_str))
        # 如果时间没有时区信息，假定为 UTC
        if dt.tzinfo is None:
            dt = pytz.UTC.localize(dt)
        else:
            dt = dt.astimezone(pytz.UTC)
        return dt
    except Exception:
        return pd.NaT

def parse_truth_time(ts_str):
    """
    解析 Truth Social 日期字符串（ISO8601，通常带 Z = UTC）。
    """
    if pd.isna(ts_str):
        return pd.NaT
    try:
        dt = date_parser.parse(str(ts_str))
        if dt.tzinfo is None:
            dt = pytz.UTC.localize(dt)
        else:
            dt = dt.astimezone(pytz.UTC)
        return dt
    except Exception:
        return pd.NaT


# 统一 published_at_utc（两个数据源均转 UTC）
df_tw["published_at_utc"] = df_tw["published_at_utc"].apply(parse_twitter_time)
df_ts["published_at_utc"] = df_ts["published_at_utc"].apply(parse_truth_time)

print(f"[Twitter] 时间解析后 UTC 范围：{df_tw['published_at_utc'].min()} ~ {df_tw['published_at_utc'].max()}")
print(f"[Truth Social] 时间解析后 UTC 范围：{df_ts['published_at_utc'].min()} ~ {df_ts['published_at_utc'].max()}")

# ============================================================
# Step 3：映射到 US/Eastern 并判定交易时段
# ============================================================
US_EASTERN = pytz.timezone("US/Eastern")

def utc_to_eastern(dt_utc):
    """UTC → US/Eastern（自动处理 DST）"""
    if pd.isna(dt_utc):
        return pd.NaT
    return dt_utc.astimezone(US_EASTERN)

def is_market_hours(dt_eastern):
    """
    判断美东时间是否在交易时段 09:30–16:00 内。
    仅对工作日（Mon–Fri）有效。
    """
    if pd.isna(dt_eastern):
        return False
    # 必须是工作日
    if dt_eastern.weekday() >= 5:   # Sat=5, Sun=6
        return False
    hour   = dt_eastern.hour
    minute = dt_eastern.minute
    # 09:30 = 9*60+30=570, 16:00 = 16*60=960
    total_minutes = hour * 60 + minute
    return 570 <= total_minutes < 960


# 转换两个数据源到美东时间
df_tw["published_at_est"] = df_tw["published_at_utc"].apply(utc_to_eastern)
df_ts["published_at_est"] = df_ts["published_at_utc"].apply(utc_to_eastern)

# 标记是否在交易时段
df_tw["in_market_hours"] = df_tw["published_at_est"].apply(is_market_hours)
df_ts["in_market_hours"] = df_ts["published_at_est"].apply(is_market_hours)

print(f"\n[Twitter] 交易时段内发布占比：{df_tw['in_market_hours'].mean():.2%}")
print(f"[Truth Social] 交易时段内发布占比：{df_ts['in_market_hours'].mean():.2%}")

# ============================================================
# 生成 NYSE 交易日骨架表（calendar spine）
# ============================================================
nyse = mcal.get_calendar("NYSE")
trading_days = nyse.valid_days(start_date=CAL_START, end_date=CAL_END)

# 转换为美东时间凌晨 0 点的交易日 Series（作为日期主键）
trading_days_series = (
    pd.to_datetime(trading_days)
    .tz_convert(US_EASTERN)
    .normalize()   # 去掉时间部分，只留日期
)

def next_trade_day(date_est, trading_days_ser, strict=False):
    """
    给定一个美东时间 date，返回其在交易日序列中的后续交易日。
    - strict=False：若 date 本身是交易日，则允许返回 date
    - strict=True：始终返回严格晚于 date 的下一个交易日
    """
    if pd.isna(date_est):
        return pd.NaT
    date_only = pd.Timestamp(date_est).normalize()
    valid = trading_days_ser[trading_days_ser > date_only] if strict else trading_days_ser[trading_days_ser >= date_only]
    if len(valid) == 0:
        return pd.NaT
    return valid[0]

def prev_trade_day(date_est, trading_days_ser):
    """
    给定一个美东时间 date，返回其在交易日序列中的上一个交易日（T-1 对齐用）。
    若 date 本身是交易日，返回 date。
    若 date 是非交易日，顺延到最近的前一个交易日。
    """
    if pd.isna(date_est):
        return pd.NaT
    date_only = pd.Timestamp(date_est).normalize()
    # 找第一个 <= date_only 的交易日
    valid = trading_days_ser[trading_days_ser <= date_only]
    if len(valid) == 0:
        return pd.NaT
    return valid.iloc[0]

print(f"\nNYSE 交易日骨架表生成完毕：{len(trading_days_series)} 个交易日")
print(f"  范围：{trading_days_series.min().date()} ~ {trading_days_series.max().date()}")

[Twitter] 时间解析后 UTC 范围：2009-05-04 18:54:25+00:00 ~ 2021-01-08 15:44:28+00:00
[Truth Social] 时间解析后 UTC 范围：2022-02-14 15:54:32.528000+00:00 ~ 2025-10-25 22:15:50.076000+00:00

[Twitter] 交易时段内发布占比：28.37%
[Truth Social] 交易时段内发布占比：30.05%

NYSE 交易日骨架表生成完毕：4780 个交易日
  范围：2008-01-01 ~ 2026-12-30


# Cell 6: market_effective_date + term_id 赋值
## market_effective_date 映射规则（与 Cell 5 的 in_market_hours 联动）：
- **in_market_hours = True**（09:30–16:00 ET 且工作日）→ market_effective_date = 当日
- **in_market_hours = False**（盘后/开盘前/周末/休市）→ market_effective_date = 下一交易日

## term_id 赋值规则：
- 2017-01-20 至 2021-01-08 UTC → `first_term`
- 2025-01-20 至 2025-10-25 UTC → `second_term`
- 其余时间 → `out_of_term`（背景分析用，不进训练池）

In [7]:
# ============================================================
# Part A：market_effective_date
# ============================================================
def get_market_effective_date(row, trading_days_ser):
    """
    推文发布 → 映射到对应交易日。
    - 交易时段内（09:30–16:00 ET 工作日）→ 当日
    - 盘后/开盘前/周末/休市 → 下一交易日
    """
    if row["in_market_hours"]:
        return pd.Timestamp(row["published_at_est"]).normalize()
    else:
        return next_trade_day(row["published_at_est"], trading_days_ser, strict=True)

df_tw["market_effective_date"] = df_tw.apply(
    lambda row: get_market_effective_date(row, trading_days_series), axis=1
)
df_ts["market_effective_date"] = df_ts.apply(
    lambda row: get_market_effective_date(row, trading_days_series), axis=1
)

print("[Twitter] market_effective_date 示例：")
print(df_tw[["published_at_est", "in_market_hours", "market_effective_date"]].head())
print(f"\n[Truth Social] market_effective_date 示例：")
print(df_ts[["published_at_est", "in_market_hours", "market_effective_date"]].head())

# ============================================================
# Part B：term_id 赋值
# ============================================================
def assign_term_id(dt_utc):
    """根据 UTC 时间戳判定 term_id"""
    if pd.isna(dt_utc):
        return "unknown"
    if FIRST_TERM_START <= dt_utc <= FIRST_TERM_END:
        return "first_term"
    if SECOND_TERM_START <= dt_utc <= SECOND_TERM_END:
        return "second_term"
    return "out_of_term"

df_tw["term_id"] = df_tw["published_at_utc"].apply(assign_term_id)
df_ts["term_id"] = df_ts["published_at_utc"].apply(assign_term_id)

print(f"\n[Twitter] term_id 分布：\n{df_tw['term_id'].value_counts()}")
print(f"\n[Truth Social] term_id 分布：\n{df_ts['term_id'].value_counts()}")

[Twitter] market_effective_date 示例：
           published_at_est  in_market_hours     market_effective_date
0 2011-08-02 14:07:48-04:00             True 2011-08-02 00:00:00-04:00
1 2020-03-02 20:34:50-05:00            False 2020-03-03 00:00:00-05:00
2 2020-01-16 22:22:47-05:00            False 2020-01-20 00:00:00-05:00
3 2020-09-12 16:10:58-04:00            False 2020-09-13 00:00:00-04:00
4 2020-01-17 08:13:59-05:00            False 2020-01-20 00:00:00-05:00

[Truth Social] market_effective_date 示例：
                  published_at_est  in_market_hours     market_effective_date
0 2025-10-25 18:15:50.076000-04:00            False 2025-10-26 00:00:00-04:00
1 2025-10-25 17:43:11.929000-04:00            False 2025-10-26 00:00:00-04:00
2 2025-10-25 17:39:14.264000-04:00            False 2025-10-26 00:00:00-04:00
3 2025-10-25 17:38:57.913000-04:00            False 2025-10-26 00:00:00-04:00
4 2025-10-25 17:38:39.636000-04:00            False 2025-10-26 00:00:00-04:00

[Twitter] term_id 分布：
term_

# Cell 6b: feature_alignment_date（T-1 最近交易日）
## 用途：Stage 1 特征关联（Lag-1 Alignment）

**规则**：
- 推文在 T 日发布 → 用 T-1 日最近有效交易日的市场特征
- 若 T-1 本身是交易日 → feature_alignment_date = T-1
- 若 T-1 是非交易日（周末/休市）→ 顺延到最近的前一个交易日

**与 market_effective_date 的区别**：
| 字段 | 用途 | 说明 |
|---|---|---|
| `market_effective_date` | 标签窗口对齐 | 事件 → 标签窗口映射 |
| `feature_alignment_date` | 宏观特征关联 | T-1 市场特征 → Stage 1 预测 |

In [8]:
# ============================================================
# feature_alignment_date：T-1 最近交易日（用于宏观特征关联）
# ============================================================
def get_feature_alignment_date(market_eff_date, trading_days_ser):
    """
    market_effective_date → T-1 最近交易日。
    规则：
    - 若 market_eff_date 为交易日 → 找其前一个交易日
    - 若前一个也是交易日 → 直接返回
    - 处理连续非交易日（如国庆长假）的情况
    """
    if pd.isna(market_eff_date):
        return pd.NaT
    # 找第一个 < market_eff_date 的交易日
    valid = trading_days_ser[trading_days_ser < market_eff_date]
    if len(valid) == 0:
        return pd.NaT
    return valid[-1]   # 最近的前一个交易日

# 1. 执行转换
df_tw["feature_alignment_date"] = df_tw["market_effective_date"].apply(
    lambda d: get_feature_alignment_date(d, trading_days_series)
)
df_ts["feature_alignment_date"] = df_ts["market_effective_date"].apply(
    lambda d: get_feature_alignment_date(d, trading_days_series)
)

# 2. 【这里是方法一】替换掉原来的 assert
# 逻辑：只对非空（notna）的行进行校验，规避 NaT 比较返回 False 的问题
mask_tw = df_tw["feature_alignment_date"].notna() & df_tw["market_effective_date"].notna()
assert (df_tw.loc[mask_tw, "feature_alignment_date"] <= df_tw.loc[mask_tw, "market_effective_date"]).all(), \
    "Twitter: feature_alignment_date 有超过 market_effective_date 的行！"

mask_ts = df_ts["feature_alignment_date"].notna() & df_ts["market_effective_date"].notna()
assert (df_ts.loc[mask_ts, "feature_alignment_date"] <= df_ts.loc[mask_ts, "market_effective_date"]).all(), \
    "Truth Social: feature_alignment_date 有超过 market_effective_date 的行！"

# 3. 验证通过后的输出
print("[feature_alignment_date] 验证通过：有效数据均符合 ≤ market_effective_date")
print(f"\n[Twitter] 示例：\n{df_tw[['published_at_est', 'market_effective_date', 'feature_alignment_date']].head(8)}")

[feature_alignment_date] 验证通过：有效数据均符合 ≤ market_effective_date

[Twitter] 示例：
           published_at_est     market_effective_date  \
0 2011-08-02 14:07:48-04:00 2011-08-02 00:00:00-04:00   
1 2020-03-02 20:34:50-05:00 2020-03-03 00:00:00-05:00   
2 2020-01-16 22:22:47-05:00 2020-01-20 00:00:00-05:00   
3 2020-09-12 16:10:58-04:00 2020-09-13 00:00:00-04:00   
4 2020-01-17 08:13:59-05:00 2020-01-20 00:00:00-05:00   
5 2020-01-16 19:11:56-05:00 2020-01-20 00:00:00-05:00   
6 2020-02-01 11:14:02-05:00 2020-02-02 00:00:00-05:00   
7 2020-10-23 00:52:14-04:00 2020-10-25 00:00:00-04:00   

     feature_alignment_date  
0 2011-08-01 00:00:00-04:00  
1 2020-03-02 00:00:00-05:00  
2 2020-01-16 00:00:00-05:00  
3 2020-09-10 00:00:00-04:00  
4 2020-01-16 00:00:00-05:00  
5 2020-01-16 00:00:00-05:00  
6 2020-01-30 00:00:00-05:00  
7 2020-10-22 00:00:00-04:00  


# Cell 7: 去重 + 合并
## 去重键规则（优先级）：
1. **主键**：`source + text_id`
2. **备选键**：`published_at_utc + text_raw_hash`（当 text_id 缺失或重复时）

## 哈希鲁棒性处理（极简归一化，防跨平台空白符差异污染去重）：
- 计算 hash 前先做极简归一化（统一换行符、逐行 strip）
- 避免同一内容因 Twitter/Truth Social 缩进不同而产生不同哈希

## 合并策略：
- 两源 concat 后按主键去重
- 优先保留 text_raw 较长的那条（减少文本截断风险）
- 保留统一后的所有元数字段

In [9]:
import hashlib
import pandas as pd

# 1. 定义归一化与哈希函数
def generate_robust_hash(text):
    """
    实现方案中的极简归一化：统一换行符、去除首尾空白、转字符串
    """
    if pd.isna(text) or text is None:
        return None
    # 极简归一化处理
    clean_text = str(text).replace('\r\n', '\n').replace('\r', '\n').strip()
    return hashlib.md5(clean_text.encode("utf-8")).hexdigest()[:16]

def raw_text_hash(text):
    """不做归一化的原始文本哈希（仅用于对比验证）"""
    if pd.isna(text) or text is None:
        return None
    return hashlib.md5(str(text).encode("utf-8")).hexdigest()[:16]

# 2. 【核心修复】：先在 df_tw 中生成归一化哈希列
# 这样 row["text_hash"] 才不会报 KeyError
df_tw['text_hash'] = df_tw['text_raw'].apply(generate_robust_hash)

# 3. 执行验证逻辑
hash_diff_count = sum(
    1 for _, row in df_tw.iterrows()
    if raw_text_hash(row["text_raw"]) != row["text_hash"]
)

print(f"[Twitter] 空白符差异导致哈希变化：{hash_diff_count} / {len(df_tw)}（占比 {hash_diff_count/len(df_tw):.4%}）")
print(f"  → 若 > 0，说明归一化成功识别了仅有空白符差异的样本")


[Twitter] 空白符差异导致哈希变化：0 / 56571（占比 0.0000%）
  → 若 > 0，说明归一化成功识别了仅有空白符差异的样本


In [10]:
# 1. 为 Truth Social 数据生成归一化哈希列
df_ts['text_hash'] = df_ts['text_raw'].apply(generate_robust_hash)

In [11]:
# --- Step 1: 预清洗 NaN (解决问题 2) ---
# FinBERT 无法处理空文本，直接剔除
df_master = pd.concat([df_tw, df_ts], ignore_index=True)
initial_count = len(df_master)
df_master = df_master.dropna(subset=['text_raw'])
print(f"剔除空文本样本：{initial_count - len(df_master)} 条")

# --- Step 2: 准备排序字段（优先保留长文本） ---
df_master['text_len'] = df_master['text_raw'].str.len()
# 按时间升序，按长度降序
df_master = df_master.sort_values(by=['published_at_utc', 'text_len'], ascending=[True, False])

# --- Step 3: 优先级去重 (解决问题 1) ---

# 1. 物理去重：source + text_id (仅在 text_id 存在时有效)
# 先处理有 ID 的部分
mask_has_id = df_master['text_id'].notna()
df_with_id = df_master[mask_has_id].drop_duplicates(subset=['source', 'text_id'], keep='first')
df_no_id = df_master[~mask_has_id]

# 重新合并
df_master = pd.concat([df_with_id, df_no_id], ignore_index=True)
print(f"物理去重后样本量：{len(df_master)}")

# 2. 语义去重：published_at_utc + text_hash
# 即使 ID 不同，如果同一秒发了同样的内容，也视为重复
df_master = df_master.drop_duplicates(subset=['published_at_utc', 'text_hash'], keep='first')
print(f"最终语义去重后样本量：{len(df_master)}")

# --- Step 4: 清理辅助列 ---
df_master = df_master.drop(columns=['text_len'])

剔除空文本样本：5053 条
物理去重后样本量：80927
最终语义去重后样本量：80927


# Cell 8: 标签窗口字段产出
## 产出的时间窗口字段：

| 字段 | 定义 | 用途 |
|---|---|---|
| `label_window_end_48h` | published_at_utc + 48h | 48h 事件语义窗 |
| `label_window_end_t2` | 48h 后下一个交易日 | purge=2 交易日对齐 |
| `market_effective_date` | 已在 Cell 6 产出 | 交易日主键 |
| `feature_alignment_date` | 已在 Cell 6b 产出 | T-1 宏观特征对齐 |

## label_window_end_t2 的 purge=2 含义：
- Stage 1 标签窗口为 2 个交易日
- embargo = 5 个交易日（外层嵌套 CV 隔离边界）
- 本字段用于验证时确认标签窗口与 purge 约束不冲突

In [12]:
# ============================================================
# label_window_end_48h
# ============================================================
df_master["label_window_end_48h"] = df_master["published_at_utc"] + pd.Timedelta(hours=48)

# ============================================================
# label_window_end_t2：48h 后的下一个交易日
# ============================================================
def get_t2_trade_day(label_window_end_utc, trading_days_ser):
    """
    给定 48h 后的 UTC 时间，找到对应的下一个交易日。
    逻辑：先转美东时间（去掉时间），再找 > 该日期的第一个交易日。
    """
    if pd.isna(label_window_end_utc):
        return pd.NaT
    # UTC → 美东日期
    date_eastern = label_window_end_utc.astimezone(US_EASTERN).normalize()
    valid = trading_days_ser[trading_days_ser > date_eastern]
    if len(valid) == 0:
        return pd.NaT
    return valid[0]

df_master["label_window_end_t2"] = df_master["label_window_end_48h"].apply(
    lambda d: get_t2_trade_day(d, trading_days_series)
)

# ============================================================
# 验证：label_window_end_t2 >= market_effective_date
# ============================================================
valid_count = (df_master["label_window_end_t2"] >= df_master["market_effective_date"]).sum()
print(f"[验证] label_window_end_t2 >= market_effective_date：{valid_count}/{len(df_master)} ({valid_count/len(df_master):.2%}）")

# ============================================================
# 预览标签窗口字段
# ============================================================
print(f"\n标签窗口字段预览：")
print(df_master[[
    "published_at_utc", "market_effective_date",
    "label_window_end_48h", "label_window_end_t2",
    "feature_alignment_date", "term_id", "source"
]].head(10).to_string())

[验证] label_window_end_t2 >= market_effective_date：80927/80927 (100.00%）

标签窗口字段预览：
           published_at_utc     market_effective_date      label_window_end_48h       label_window_end_t2    feature_alignment_date      term_id   source
0 2009-05-04 18:54:25+00:00 2009-05-04 00:00:00-04:00 2009-05-06 18:54:25+00:00 2009-05-07 00:00:00-04:00 2009-05-03 00:00:00-04:00  out_of_term  twitter
1 2009-05-05 01:00:10+00:00 2009-05-05 00:00:00-04:00 2009-05-07 01:00:10+00:00 2009-05-07 00:00:00-04:00 2009-05-04 00:00:00-04:00  out_of_term  twitter
2 2009-05-08 13:38:08+00:00 2009-05-08 00:00:00-04:00 2009-05-10 13:38:08+00:00 2009-05-11 00:00:00-04:00 2009-05-07 00:00:00-04:00  out_of_term  twitter
3 2009-05-08 20:40:15+00:00 2009-05-10 00:00:00-04:00 2009-05-10 20:40:15+00:00 2009-05-11 00:00:00-04:00 2009-05-07 00:00:00-04:00  out_of_term  twitter
4 2009-05-12 14:07:28+00:00 2009-05-12 00:00:00-04:00 2009-05-14 14:07:28+00:00 2009-05-17 00:00:00-04:00 2009-05-11 00:00:00-04:00  out_of_term  t

# Cell 9: 输出结果 + 质量报告
## 输出文件清单：

| 文件名 | 内容 | 主要字段 |
|---|---|---|
| `01_text_master.csv` | 统一文本主表（逐条级） | text_id, source, text_raw, text_clean, published_at_utc, term_id, market_effective_date, feature_alignment_date, is_retweet, label_window_end_48h, label_window_end_t2 |
| `02_event_windows.csv` | 事件窗口中间表 | text_id, market_effective_date, feature_alignment_date, label_window_end_48h, label_window_end_t2 |
| `03_daily_features.csv` | 日频聚合表（预留宏观对齐接口） | market_effective_date, tweet_count, retweet_count, term_id breakdown |
| `04_quality_report.csv` | 质量报告 | 各维度缺失率、重复率、编码问题率、孤立事件率、term × source 分层 |

## 质量报告维度：
- 缺失率（按字段）
- 重复率（source + text_id 维度）
- 编码问题率（mojibake 修复前后对比）
- **孤立事件率**：某推文发出后 48h 内同 source+term_id 组合下无其他推文的比例——这类样本 LLM 难以判定 TACO，需标记为 Low Confidence
- term_id × source 分层样本统计
- 周末/盘后发布占比（影响 market_effective_date 顺延比例）

In [13]:
# ============================================================
# Part D：输出 04_quality_report.csv
# ============================================================
quality_rows = []

# 按 source 分层质量指标
for src, df_src in df_master.groupby("source"):
    n = len(df_src)
    quality_rows.append({
        "dimension": "source",
        "value": src,
        "total_rows": n,
        "missing_text_raw": df_src["text_raw"].isna().sum(),
        "missing_text_clean": df_src["text_clean"].isna().sum(),
        "empty_after_clean": (df_src["text_clean"] == "").sum(),
        "retweet_ratio": df_src["is_retweet"].mean(),
        "missing_market_eff_date": df_src["market_effective_date"].isna().sum(),
        "missing_feature_align_date": df_src["feature_alignment_date"].isna().sum(),
        "off_market_hours_ratio": (~df_src["in_market_hours"]).mean(),
    })

# 按 term_id 分层质量指标
for tid, df_tid in df_master.groupby("term_id"):
    n = len(df_tid)
    quality_rows.append({
        "dimension": "term_id",
        "value": tid,
        "total_rows": n,
        "missing_text_raw": df_tid["text_raw"].isna().sum(),
        "missing_text_clean": df_tid["text_clean"].isna().sum(),
        "empty_after_clean": (df_tid["text_clean"] == "").sum(),
        "retweet_ratio": df_tid["is_retweet"].mean(),
        "missing_market_eff_date": df_tid["market_effective_date"].isna().sum(),
        "missing_feature_align_date": df_tid["feature_alignment_date"].isna().sum(),
        "off_market_hours_ratio": (~df_tid["in_market_hours"]).mean(),
    })

# 总体
n_total = len(df_master)
quality_rows.append({
    "dimension": "overall",
    "value": "all",
    "total_rows": n_total,
    "missing_text_raw": df_master["text_raw"].isna().sum(),
    "missing_text_clean": df_master["text_clean"].isna().sum(),
    "empty_after_clean": (df_master["text_clean"] == "").sum(),
    "retweet_ratio": df_master["is_retweet"].mean(),
    "missing_market_eff_date": df_master["market_effective_date"].isna().sum(),
    "missing_feature_align_date": df_master["feature_alignment_date"].isna().sum(),
    "off_market_hours_ratio": (~df_master["in_market_hours"]).mean(),
})

df_quality = pd.DataFrame(quality_rows)

# 数值列转比率
ratio_cols = ["missing_text_raw", "missing_text_clean", "empty_after_clean",
              "missing_market_eff_date", "missing_feature_align_date"]
for col in ratio_cols:
    df_quality[f"{col}_rate"] = df_quality[col] / df_quality["total_rows"]

# ============================================================
# Part D2：孤立事件率（isolated_event_ratio）
# "孤立事件"定义：某推文发出后 48h 内，同一 source + term_id 组合下无其他推文
# 这类样本 LLM 难以判定 TACO（无后续可对比），需标记为 Low Confidence
# ============================================================
def compute_isolated_ratio(df_group):
    """
    对一个 (source, term_id) 分组，计算孤立事件率。
    逻辑：在 label_window_end_48h 之前/之内，同 source+term_id 分组内只有自己一条。
    """
    if len(df_group) == 0:
        return 0.0
    n = len(df_group)
    isolated = 0
    for _, row in df_group.iterrows():
        window_end = row["label_window_end_48h"]
        same_group = (
            (df_group["source"] == row["source"]) &
            (df_group["term_id"] == row["term_id"]) &
            (df_group["published_at_utc"] != row["published_at_utc"]) &
            (df_group["published_at_utc"] <= window_end)
        )
        if same_group.sum() == 0:
            isolated += 1
    return isolated / n

print("[孤立事件分析] 正在计算（按 source × term_id 分组）...")
isolated_ratios = {}
group_sizes = df_master.groupby(["source", "term_id"]).size().to_dict()
for (src, tid), grp in df_master.groupby(["source", "term_id"]):
    ratio = compute_isolated_ratio(grp)
    isolated_ratios[(src, tid)] = ratio
    print(f"  {src} × {tid}: 孤立事件率 {ratio:.2%}")

# 加入 quality_report
for row in quality_rows:
    if row["dimension"] == "source":
        matched = [isolated_ratios[(src, tid)] * group_sizes[(src, tid)] for (src, tid) in isolated_ratios if src == row["value"]]
        denom = sum(group_sizes[(src, tid)] for (src, tid) in isolated_ratios if src == row["value"])
        row["isolated_event_ratio"] = sum(matched) / denom if denom > 0 else 0.0
    elif row["dimension"] == "term_id":
        matched = [isolated_ratios[(src, tid)] * group_sizes[(src, tid)] for (src, tid) in isolated_ratios if tid == row["value"]]
        denom = sum(group_sizes[(src, tid)] for (src, tid) in isolated_ratios if tid == row["value"])
        row["isolated_event_ratio"] = sum(matched) / denom if denom > 0 else 0.0
    else:  # overall
        # overall = 所有 source × term_id 分组的加权平均
        total_n = sum(group_sizes.values())
        weighted_sum = sum(
            isolated_ratios.get((src, tid), 0) * group_sizes[(src, tid)]
            for (src, tid) in isolated_ratios
        )
        row["isolated_event_ratio"] = weighted_sum / total_n if total_n > 0 else 0.0

df_quality = pd.DataFrame(quality_rows)

df_quality.to_csv(
    DATA_OUT / "04_quality_report.csv",
    index=False,
    encoding="utf-8"
)
print(f"[OK] 04_quality_report.csv 已保存（含孤立事件率）")

[孤立事件分析] 正在计算（按 source × term_id 分组）...
  truthsocial × out_of_term: 孤立事件率 0.00%
  truthsocial × second_term: 孤立事件率 0.00%
  twitter × first_term: 孤立事件率 0.00%
  twitter × out_of_term: 孤立事件率 0.00%
[OK] 04_quality_report.csv 已保存（含孤立事件率）


In [14]:
# 01_text_master.csv
df_master[[
    "text_id", "source", "text_raw", "text_clean", "published_at_utc",
    "url", "term_id", "market_effective_date", "feature_alignment_date",
    "is_retweet", "is_trump_direct_text", "label_window_end_48h", "label_window_end_t2"
]].to_csv(DATA_OUT / "01_text_master.csv", index=False, encoding="utf-8")

# 02_event_windows.csv
df_master[[
    "text_id", "market_effective_date", "feature_alignment_date",
    "label_window_end_48h", "label_window_end_t2"
]].to_csv(DATA_OUT / "02_event_windows.csv", index=False, encoding="utf-8")

# 03_daily_features.csv（按交易日聚合）
daily_agg = df_master.groupby("market_effective_date").agg(
    tweet_count=("text_id", "count"),
    retweet_count=("is_retweet", "sum"),
    first_term_count=("term_id", lambda x: (x == "first_term").sum()),
    second_term_count=("term_id", lambda x: (x == "second_term").sum()),
    out_of_term_count=("term_id", lambda x: (x == "out_of_term").sum()),
).reset_index()
daily_agg.to_csv(DATA_OUT / "03_daily_features.csv", index=False, encoding="utf-8")


In [15]:
# 按 term_id 分三个 CSV 输出，并只保留第一任期开始后的 Trump direct text 作为建模主输入
direct_master = df_master[
    (df_master['is_trump_direct_text']) &
    (df_master['published_at_utc'] >= FIRST_TERM_START)
].copy()
first_term_df = direct_master[direct_master['term_id'] == 'first_term']
second_term_df = direct_master[direct_master['term_id'] == 'second_term']
out_of_term_df = direct_master[direct_master['term_id'] == 'out_of_term']

first_term_df.to_csv(DATA_OUT / "01_first_term.csv", index=False)
second_term_df.to_csv(DATA_OUT / "02_second_term.csv", index=False)
out_of_term_df.to_csv(DATA_OUT / "03_out_of_term.csv", index=False)


## Part 2. Text Preparation for Modeling

在主表基础上补充适合建模与情绪分析的文本字段。

# Text Cleaning for TACO Datasets

在 `01_data_cleaning.ipynb` 已完成的最小清洗基础上补充 `text_finbert`，避免二次清洗把 HTML/编码修复回退掉。

| 列名 | 处理方式 | 用途 |
|------|----------|------|
| `text_raw` | 原始备份（含 URL 和 Emoji） | 查源、人工审计 |
| `text_clean` | 继承上游最小清洗结果 | DeepSeek 标注 |
| `text_finbert` | 基于 `text_clean` 移除 Emoji | FinBERT 提取 |
| `url` | 独立 URL 链接列 | 快速点击回溯 |

In [14]:
import re
import html
import pandas as pd
import ftfy

DATA_DIR = "/Users/horace/Coding/ML finance/project/data/processed"
FILES = {
    "first_term": f"{DATA_DIR}/01_first_term.csv",
    "second_term": f"{DATA_DIR}/02_second_term.csv",
    "out_of_term": f"{DATA_DIR}/03_out_of_term.csv",
}

In [15]:
URL_PATTERN = re.compile(r'https?://\\S+|www\\.\\S+', re.IGNORECASE)

EMOJI_PATTERN = re.compile(
    "["
    "\\U0001F600-\\U0001F64F"  # emoticons
    "\\U0001F300-\\U0001F5FF"  # symbols & pictographs
    "\\U0001F680-\\U0001F6FF"  # transport & map symbols
    "\\U0001F1E0-\\U0001F1FF"  # flags
    "\\U00002702-\\U000027B0"  # dingbats
    "\\U000024C2-\\U0001F251"  # enclosed characters
    "]+",
    flags=re.UNICODE
)


def ensure_minimal_clean(text):
    """仅在缺失上游 text_clean 时兜底，逻辑与 01_data_cleaning 保持一致。"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = html.unescape(text)
    text = ftfy.fix_text(text)
    text = URL_PATTERN.sub('[URL]', text)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\uFFFD]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def clean_text_finbert(text):
    """基于 text_clean 生成 FinBERT 输入，只额外移除 Emoji。"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    text = EMOJI_PATTERN.sub('', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def extract_url(text):
    """从 text_raw 中提取第一个 URL。"""
    if pd.isna(text) or not isinstance(text, str):
        return ""
    match = URL_PATTERN.search(text)
    return match.group(0) if match else ""

In [16]:
# 加载并处理每个文件
for name, path in FILES.items():
    print(f"\n{'='*60}")
    print(f"处理: {name}")
    print(f"路径: {path}")
    print('='*60)

    df = pd.read_csv(path)
    print(f"行数: {len(df)}, 列: {list(df.columns)}")

    if "text_clean" not in df.columns:
        df["text_clean"] = df["text_raw"].apply(ensure_minimal_clean)
    else:
        missing_mask = df["text_clean"].isna() | (df["text_clean"].astype(str).str.strip() == "")
        df.loc[missing_mask, "text_clean"] = df.loc[missing_mask, "text_raw"].apply(ensure_minimal_clean)

    df["text_finbert"] = df["text_clean"].apply(clean_text_finbert)

    if "url" not in df.columns:
        df["url"] = pd.Series("", index=df.index, dtype="string")
    else:
        df["url"] = df["url"].astype("string")
    missing_url = df["url"].isna() | (df["url"].str.strip() == "")
    df.loc[missing_url, "url"] = df.loc[missing_url, "text_raw"].apply(extract_url).astype("string")

    df.to_csv(path, index=False)
    print(f"已保存: {path}")

    print("\n样本对比:")
    sample = df[["text_raw", "text_clean", "text_finbert", "url"]].head(3)
    for i, row in sample.iterrows():
        print(f"\n--- 样本 {i} ---")
        print(f"text_raw:    {row['text_raw'][:100]}...")
        print(f"text_clean:  {row['text_clean'][:100]}...")
        print(f"text_finbert:{row['text_finbert'][:100]}...")
        print(f"url:         {row['url']}")


处理: first_term
路径: /Users/horace/Coding/ML finance/project/data/processed/01_first_term.csv
行数: 16581, 列: ['text_id', 'text_raw', 'published_at_utc', 'device', 'is_retweet', 'is_trump_direct_text', 'source', 'text_clean', 'published_at_est', 'in_market_hours', 'market_effective_date', 'term_id', 'feature_alignment_date', 'text_hash', 'url', 'label_window_end_48h', 'label_window_end_t2']
已保存: /Users/horace/Coding/ML finance/project/data/processed/01_first_term.csv

样本对比:

--- 样本 0 ---
text_raw:    Today we are not merely transferring power from one Administration to another, or from one party to ...
text_clean:  Today we are not merely transferring power from one Administration to another, or from one party to ...
text_finbert:Today we are not merely transferring power from one Administration to another, or from one party to ...
url:         

--- 样本 1 ---
text_raw:    power from Washington, D.C. and giving it back to you, the American People. #InaugurationDay...
text_clean:  power fro

In [17]:
# 验证：检查 Emoji 处理情况
print("\n检查 Emoji 处理情况（text_clean 保留，text_finbert 移除）:")
for name, path in FILES.items():
    df = pd.read_csv(path)
    emoji_rows = df[df["text_clean"].apply(lambda x: bool(EMOJI_PATTERN.search(str(x))))]
    print(f"\n{name}: {len(emoji_rows)} 行包含 Emoji")

    if len(emoji_rows) > 0:
        row = emoji_rows.iloc[0]
        print(f"  text_raw:    {row['text_raw'][:80]}...")
        print(f"  text_clean:  {row['text_clean'][:80]}... (Emoji 保留)")
        print(f"  text_finbert:{row['text_finbert'][:80]}... (Emoji 已移除)")


检查 Emoji 处理情况（text_clean 保留，text_finbert 移除）:

first_term: 264 行包含 Emoji
  text_raw:    The forgotten men and women of our country will be forgotten no longer. From thi...
  text_clean:  The forgotten men and women of our country will be forgotten no longer. From thi... (Emoji 保留)
  text_finbert:The forgotten men and women of our country will be forgotten no longer. From thi... (Emoji 已移除)

second_term: 25 行包含 Emoji
  text_raw:    WELCOME HOME, MARC FOGEL🇺🇸...
  text_clean:  WELCOME HOME, MARC FOGEL🇺🇸... (Emoji 保留)
  text_finbert:WELCOME HOME, MARC FOGEL... (Emoji 已移除)

out_of_term: 92 行包含 Emoji
  text_raw:    𝗠𝗔𝗞𝗘 𝗔𝗠𝗘𝗥𝗜𝗖𝗔 𝗚𝗥𝗘𝗔𝗧 𝗔𝗚𝗔𝗜𝗡!...
  text_clean:  𝗠𝗔𝗞𝗘 𝗔𝗠𝗘𝗥𝗜𝗖𝗔 𝗚𝗥𝗘𝗔𝗧 𝗔𝗚𝗔𝗜𝗡!... (Emoji 保留)
  text_finbert:!... (Emoji 已移除)


## Part 3. Candidate Event Generation

基于规则生成候选事件，并构造 48 小时跟踪上下文。

# Candidate Event Generation

This notebook builds an `event-level` candidate set for Stage 1 labeling.

Pipeline:
1. Load first-term and second-term cleaned text tables
2. Apply high-recall, rule-first candidate filters
3. Build 48h follow-up context for each anchor text
4. Prefer relevant follow-up lines and fall back to the full 48h window only when needed
5. Export `data/processed/03_candidate_events.csv`


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_PATHS = [
    ROOT / "data/processed/01_first_term.csv",
    ROOT / "data/processed/02_second_term.csv",
]
OUTPUT_PATH = ROOT / "data/processed/03_candidate_events.csv"

frames = [pd.read_csv(path, low_memory=False) for path in INPUT_PATHS]
text_df = pd.concat(frames, ignore_index=True)

text_df["published_at_utc"] = pd.to_datetime(text_df["published_at_utc"], utc=True, format="mixed")
text_df["published_at_est"] = pd.to_datetime(text_df["published_at_est"], utc=True, format="mixed").dt.tz_convert("America/New_York")
text_df["label_window_end_48h"] = pd.to_datetime(text_df["label_window_end_48h"], utc=True, format="mixed")

text_df = text_df.loc[text_df["is_trump_direct_text"].fillna(False)].copy()
text_df = text_df.sort_values("published_at_utc").reset_index(drop=True)

display(text_df[["text_id", "source", "term_id", "published_at_utc", "text_clean"]].head())
print(f"Loaded {len(text_df):,} Trump direct texts.")


In [ ]:
CORE_THEME_TERMS = [
    "tariff",
    "tariffs",
    "trade",
    "china",
    "mexico",
    "canada",
    "sanction",
    "sanctions",
    "steel",
    "aluminum",
    "aluminium",
    "levy",
    "levies",
    "import",
    "imports",
    "export",
    "exports",
    "duty",
    "duties",
]

CONTEXTUAL_THEME_RULES = {
    "deal": [
        "trade",
        "tariff",
        "tariffs",
        "china",
        "mexico",
        "canada",
        "iran",
        "sanction",
        "sanctions",
        "oil",
        "nuclear",
        "import",
        "imports",
        "export",
        "exports",
    ],
    "iran": ["deal", "sanction", "sanctions", "oil", "nuclear"],
    "eu": [
        "trade",
        "tariff",
        "tariffs",
        "barrier",
        "barriers",
        "import",
        "imports",
        "export",
        "exports",
        "cars",
        "autos",
        "steel",
        "aluminum",
        "levy",
        "levies",
        "deficit",
    ],
    "european union": [
        "trade",
        "tariff",
        "tariffs",
        "barrier",
        "barriers",
        "import",
        "imports",
        "export",
        "exports",
        "cars",
        "autos",
        "steel",
        "aluminum",
        "levy",
        "levies",
        "deficit",
    ],
    "europe": [
        "trade",
        "tariff",
        "tariffs",
        "barrier",
        "barriers",
        "import",
        "imports",
        "export",
        "exports",
        "cars",
        "autos",
        "steel",
        "aluminum",
        "levy",
        "levies",
        "deficit",
    ],
}

ACTION_TERMS = [
    "raise",
    "raised",
    "impose",
    "imposed",
    "hit",
    "charge",
    "charged",
    "tax",
    "taxed",
    "must",
    "unless",
    "effective",
    "will be",
    "increase",
    "increased",
    "cancel",
    "terminated",
    "terminate",
    "put on notice",
]

OBJECT_TERMS = [
    "goods",
    "imports",
    "products",
    "cars",
    "autos",
    "automobiles",
    "border",
    "barrier",
    "barriers",
    "farmers",
    "manufacturers",
]

SENTIMENT_PRIORITY_COLS = ["p_neg", "neg_prob", "negative_score", "finbert_neg"]
TRADE_CONTEXT_TERMS = [
    "trade",
    "tariff",
    "tariffs",
    "sanction",
    "sanctions",
    "deal",
    "china",
    "mexico",
    "canada",
    "iran",
    "oil",
    "nuclear",
    "import",
    "imports",
    "export",
    "exports",
    "barrier",
    "barriers",
    "cars",
    "autos",
    "steel",
    "aluminum",
    "levy",
    "levies",
]


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip().lower()
    return re.sub(r"\s+", " ", text)


def unique_preserve(values: list[str]) -> list[str]:
    seen = set()
    ordered = []
    for value in values:
        if value not in seen:
            seen.add(value)
            ordered.append(value)
    return ordered


def collect_hits(text: str, vocab: list[str]) -> list[str]:
    normalized = normalize_text(text)
    return [term for term in vocab if term in normalized]


def collect_contextual_hits(text: str) -> list[str]:
    normalized = normalize_text(text)
    hits = []
    for term, modifiers in CONTEXTUAL_THEME_RULES.items():
        if term in normalized and any(modifier in normalized for modifier in modifiers):
            hits.append(term)
    return unique_preserve(hits)


def build_relevance_terms(row: pd.Series) -> list[str]:
    terms = []
    terms.extend(row["theme_hits"])
    terms.extend(row["action_hits"])
    terms.extend(row["object_hits"])

    normalized = normalize_text(row["rule_text"])
    if any(term in normalized for term in ["china", "mexico", "canada", "trade", "tariff", "tariffs"]):
        terms.extend(["trade", "tariff", "tariffs", "china", "mexico", "canada", "import", "imports", "export", "exports"])
    if any(term in normalized for term in ["iran", "sanction", "sanctions", "oil", "nuclear", "deal"]):
        terms.extend(["iran", "deal", "sanction", "sanctions", "oil", "nuclear"])
    if any(term in normalized for term in ["eu", "european union", "europe", "steel", "aluminum", "aluminium"]):
        terms.extend(["eu", "european union", "europe", "trade", "tariff", "tariffs", "barrier", "barriers", "cars", "steel", "aluminum"])

    return unique_preserve([term for term in terms if term])


text_df["rule_text"] = text_df["text_clean"].fillna("")
blank_clean = text_df["rule_text"].astype(str).str.strip().eq("")
text_df.loc[blank_clean, "rule_text"] = text_df.loc[blank_clean, "text_raw"].fillna("")
text_df["normalized_rule_text"] = text_df["rule_text"].map(normalize_text)

text_df["theme_hits"] = text_df["rule_text"].map(
    lambda x: unique_preserve(collect_hits(x, CORE_THEME_TERMS) + collect_contextual_hits(x))
)
text_df["action_hits"] = text_df["rule_text"].map(lambda x: collect_hits(x, ACTION_TERMS))
text_df["object_hits"] = text_df["rule_text"].map(lambda x: collect_hits(x, OBJECT_TERMS))

text_df["theme_hit_count"] = text_df["theme_hits"].str.len()
text_df["action_hit_count"] = text_df["action_hits"].str.len()
text_df["object_hit_count"] = text_df["object_hits"].str.len()

high_priority_mask = (text_df["theme_hit_count"] > 0) & (
    (text_df["action_hit_count"] > 0) | (text_df["object_hit_count"] > 0)
)
low_priority_mask = (text_df["theme_hit_count"] > 0) & ~high_priority_mask

text_df["candidate_priority"] = np.select(
    [high_priority_mask, low_priority_mask],
    ["high", "low"],
    default="exclude",
)
text_df["is_candidate"] = text_df["candidate_priority"].ne("exclude")

sentiment_col = next((col for col in SENTIMENT_PRIORITY_COLS if col in text_df.columns), None)
if sentiment_col is not None:
    print(f"Using optional sentiment priority column: {sentiment_col}")
else:
    print("No FinBERT probability column found. Candidate generation stays rule-first.")

text_df["keyword_score"] = (
    text_df["theme_hit_count"] * 2
    + text_df["action_hit_count"]
    + text_df["object_hit_count"]
)
text_df["sentiment_priority"] = text_df[sentiment_col] if sentiment_col else np.nan

display(
    text_df.loc[text_df["is_candidate"], [
        "text_id",
        "candidate_priority",
        "theme_hits",
        "action_hits",
        "object_hits",
    ]].head()
)


In [ ]:
candidate_df = text_df.loc[text_df["is_candidate"]].copy()
candidate_df["published_at_utc_naive"] = candidate_df["published_at_utc"].dt.tz_convert("UTC").dt.tz_localize(None)
candidate_df["label_window_end_48h_naive"] = candidate_df["label_window_end_48h"].dt.tz_convert("UTC").dt.tz_localize(None)
candidate_df["relevance_terms"] = candidate_df.apply(build_relevance_terms, axis=1)

timeline_df = text_df[["published_at_utc", "published_at_est", "rule_text", "normalized_rule_text"]].copy()
timeline_df = timeline_df.loc[timeline_df["rule_text"].astype(str).str.strip().ne("")].copy()
timeline_df["published_at_utc_naive"] = timeline_df["published_at_utc"].dt.tz_convert("UTC").dt.tz_localize(None)
timeline_df["context_line"] = (
    timeline_df["published_at_est"].astype(str)
    + " | "
    + timeline_df["rule_text"].astype(str).str.replace("\n", " ", regex=False)
)

timeline_times = timeline_df["published_at_utc_naive"].to_numpy(dtype="datetime64[ns]")
timeline_lines = timeline_df["context_line"].to_list()
timeline_normalized = timeline_df["normalized_rule_text"].to_list()


def build_context(anchor_time, window_end, relevance_terms):
    anchor_time = np.datetime64(anchor_time)
    window_end = np.datetime64(window_end)
    left = np.searchsorted(timeline_times, anchor_time, side="right")
    right = np.searchsorted(timeline_times, window_end, side="right")

    all_lines = timeline_lines[left:right]
    all_normalized = timeline_normalized[left:right]

    relevant_lines = []
    if relevance_terms:
        for line, normalized in zip(all_lines, all_normalized):
            if any(term in normalized for term in relevance_terms):
                relevant_lines.append(line)

    if relevant_lines:
        selected_lines = relevant_lines
        context_mode = "relevant_only"
    else:
        selected_lines = all_lines
        context_mode = "fallback_all"

    selected_text = "None" if not selected_lines else "\n\n".join(selected_lines)
    return {
        "follow_up_count_all_48h": len(all_lines),
        "follow_up_count_relevant_48h": len(relevant_lines),
        "follow_up_text_selected": selected_text,
        "context_mode": context_mode,
    }


context_records = []
for row in candidate_df[["published_at_utc_naive", "label_window_end_48h_naive", "relevance_terms"]].itertuples(index=False):
    context_records.append(build_context(row.published_at_utc_naive, row.label_window_end_48h_naive, row.relevance_terms))

context_df = pd.DataFrame(context_records)
candidate_df = pd.concat([candidate_df.reset_index(drop=True), context_df], axis=1)

candidate_df["review_flag"] = np.where(candidate_df["follow_up_count_relevant_48h"].eq(0), "yes", "no")
candidate_df["source_text"] = (
    "[Anchor]\n"
    + candidate_df["published_at_est"].astype(str)
    + " | "
    + candidate_df["rule_text"].astype(str).str.replace("\n", " ", regex=False)
    + "\n\n[Follow-up within 48h]\n"
    + candidate_df["follow_up_text_selected"]
)

candidate_df["event_id"] = (
    candidate_df["term_id"].fillna("unknown").str.upper()
    + "_"
    + candidate_df["source"].fillna("unknown").str.upper()
    + "_"
    + candidate_df["text_id"].astype(str)
)

candidate_events = candidate_df.rename(
    columns={
        "published_at_utc": "event_time",
        "published_at_est": "event_time_est",
    }
).copy()

candidate_events = candidate_events[[
    "event_id",
    "text_id",
    "source",
    "term_id",
    "event_time",
    "event_time_est",
    "market_effective_date",
    "label_window_end_48h",
    "candidate_priority",
    "keyword_score",
    "theme_hits",
    "action_hits",
    "object_hits",
    "relevance_terms",
    "follow_up_count_all_48h",
    "follow_up_count_relevant_48h",
    "context_mode",
    "review_flag",
    "rule_text",
    "source_text",
    "url",
]]

candidate_events = candidate_events.sort_values(["event_time", "event_id"]).reset_index(drop=True)
display(candidate_events.head())


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
candidate_events.to_csv(OUTPUT_PATH, index=False)

print(f"Saved {len(candidate_events):,} candidate events to {OUTPUT_PATH}")
display(candidate_events["candidate_priority"].value_counts(dropna=False).rename_axis("candidate_priority").to_frame("count"))
display(candidate_events["context_mode"].value_counts(dropna=False).rename_axis("context_mode").to_frame("count"))
display(candidate_events[[
    "event_id",
    "event_time_est",
    "candidate_priority",
    "keyword_score",
    "theme_hits",
    "action_hits",
    "follow_up_count_all_48h",
    "follow_up_count_relevant_48h",
    "context_mode",
]].head(10))


## Notes

- `deal` and `eu / europe` now require trade-policy context instead of triggering on their own.
- `iran` is preserved as a contextual theme and only activates with `deal / sanction / oil / nuclear` style cues.
- `source_text` prefers relevant follow-up lines and falls back to the full 48h window only when no relevant lines are found.


## Part 4. FinBERT Inference

对清洗后的文本运行全量 FinBERT 打分。

# FinBERT Full Inference

This notebook runs full-batch FinBERT inference on the cleaned text tables.

Target setup:
- Full dataset
- 8GB GPU preferred
- Conservative default batch size


In [1]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [2]:
from contextlib import nullcontext
from pathlib import Path
import math

import pandas as pd
from tqdm.auto import tqdm

d:\anaconda3\envs\dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_PATHS = [
    ROOT / "./01_first_term.csv",
    ROOT / "./02_second_term.csv",
]
OUTPUT_ALL = ROOT / "./processed/04_finbert_scored_all.csv"
OUTPUT_FIRST = ROOT / "./processed/04_finbert_scored_first_term.csv"
OUTPUT_SECOND = ROOT / "./processed/04_finbert_scored_second_term.csv"

BATCH_SIZE = 32  # safe starting point for 8GB GPU
MODEL_NAME = "ProsusAI/finbert"
MAX_LENGTH = 256
TEXT_COL_CANDIDATES = ["text_finbert", "text_clean", "text_raw"]

print(f"Project root: {ROOT}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Model: {MODEL_NAME}")


Project root: c:\Users\gumen\Desktop\ML-fin
Batch size: 32
Model: ProsusAI/finbert


In [4]:
try:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
except Exception as exc:
    raise RuntimeError(
        "Missing dependency. Install required packages first: pip install transformers torch"
    ) from exc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AUTOCAST = DEVICE == "cuda"
print(f"Using device: {DEVICE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
    print("Autocast enabled: True")
else:
    print("Autocast enabled: False")
    print("Full run is intended for GPU. CPU fallback is available but will be much slower.")


Using device: cuda
NVIDIA GeForce RTX 3070 Ti
Autocast enabled: True


In [5]:
frames = [pd.read_csv(path, low_memory=False) for path in INPUT_PATHS]
df = pd.concat(frames, ignore_index=True)

df = df.loc[df["is_trump_direct_text"].fillna(False)].copy()


def choose_input_text(row: pd.Series) -> str:
    for col in TEXT_COL_CANDIDATES:
        value = row.get(col)
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return ""


df["finbert_input_text"] = df.apply(choose_input_text, axis=1)
df = df.loc[df["finbert_input_text"].str.strip().ne("")].copy()
df = df.reset_index(drop=True)

print(f"Rows prepared for inference: {len(df):,}")
display(df[["text_id", "source", "term_id", "finbert_input_text"]].head())


Rows prepared for inference: 20,093


,text_id,source,term_id,finbert_input_text
0,822501803615014918,twitter,first_term,Today we are not merely transferring power fro...
1,822501939267141634,twitter,first_term,"power from Washington, D.C. and giving it back..."
2,822502135233384448,twitter,first_term,What truly matters is not which party controls...
3,822502270503972872,twitter,first_term,"January 20th 2017, will be remembered as the d..."
4,822502450007515137,twitter,first_term,The forgotten men and women of our country wil...


In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

id2label = model.config.id2label
print(id2label)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 28338.94it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{0: 'positive', 1: 'negative', 2: 'neutral'}


In [7]:
def batched(iterable, batch_size: int):
    total = len(iterable)
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        yield start, end, iterable[start:end]


def get_autocast_context():
    if USE_AUTOCAST:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    return nullcontext()


all_pos = []
all_neu = []
all_neg = []
all_label = []

texts = df["finbert_input_text"].tolist()
num_batches = math.ceil(len(texts) / BATCH_SIZE) if texts else 0

for start, end, batch_texts in tqdm(batched(texts, BATCH_SIZE), total=num_batches):
    encoded = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    encoded = {k: v.to(DEVICE) for k, v in encoded.items()}

    try:
        with torch.inference_mode():
            with get_autocast_context():
                logits = model(**encoded).logits
            probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
    except RuntimeError as exc:
        if "out of memory" in str(exc).lower():
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
            raise RuntimeError(
                f"CUDA OOM at rows {start}:{end}. Reduce BATCH_SIZE below {BATCH_SIZE} and rerun."
            ) from exc
        raise

    labels = probs.argmax(axis=1)

    for prob_row, label_idx in zip(probs, labels):
        label_map = {id2label[i].lower(): prob_row[i] for i in range(len(prob_row))}
        all_pos.append(float(label_map.get("positive", 0.0)))
        all_neu.append(float(label_map.get("neutral", 0.0)))
        all_neg.append(float(label_map.get("negative", 0.0)))
        all_label.append(id2label[int(label_idx)].lower())

df["finbert_label"] = all_label
df["finbert_pos"] = all_pos
df["finbert_neu"] = all_neu
df["finbert_neg"] = all_neg

display(df[["text_id", "finbert_label", "finbert_pos", "finbert_neu", "finbert_neg"]].head())


100%|██████████| 628/628 [00:16<00:00, 37.73it/s]


,text_id,finbert_label,finbert_pos,finbert_neu,finbert_neg
0,822501803615014918,neutral,0.063049,0.919434,0.017349
1,822501939267141634,neutral,0.051727,0.927246,0.021271
2,822502135233384448,neutral,0.029373,0.915527,0.055298
3,822502270503972872,neutral,0.126099,0.860840,0.013145
4,822502450007515137,neutral,0.040558,0.928223,0.031174


In [8]:
prob_sum = df[["finbert_pos", "finbert_neu", "finbert_neg"]].sum(axis=1)
print("Probability sum check:")
print(prob_sum.describe())

print("Label distribution:")
print(df["finbert_label"].value_counts(dropna=False))


Probability sum check:
count    20093.000000
mean         0.999982
std          0.000154
min          0.999634
25%          0.999847
50%          0.999977
75%          1.000122
max          1.000366
dtype: float64
Label distribution:
finbert_label
neutral     14762
negative     3512
positive     1819
Name: count, dtype: int64


In [9]:
OUTPUT_ALL.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_ALL, index=False)
df.loc[df["term_id"].eq("first_term")].to_csv(OUTPUT_FIRST, index=False)
df.loc[df["term_id"].eq("second_term")].to_csv(OUTPUT_SECOND, index=False)

print(f"Full output saved to: {OUTPUT_ALL}")
print(f"First-term output saved to: {OUTPUT_FIRST}")
print(f"Second-term output saved to: {OUTPUT_SECOND}")


Full output saved to: c:\Users\gumen\Desktop\ML-fin\processed\04_finbert_scored_all.csv
First-term output saved to: c:\Users\gumen\Desktop\ML-fin\processed\04_finbert_scored_first_term.csv
Second-term output saved to: c:\Users\gumen\Desktop\ML-fin\processed\04_finbert_scored_second_term.csv


## Notes

- This notebook is intended for full inference on the project dataset.
- For an 8GB GPU, start with `BATCH_SIZE = 16`.
- If stable, you can try `24`.
- If OOM occurs, reduce to `8`.
- Keep `text_id` untouched so the results can be joined back later.


## Part 5. Candidate Event Calendar Standardization

标准化候选事件时间字段，并推导交易日相关锚点。

# Candidate Event Standardization

作用：
- 读取 `data/processed/03_candidate_events.csv`
- 标准化事件时间字段
- 推导 `feature_anchor_date` 与 `target_trade_date`
- 写出 `data/processed/standardized/03_candidate_events_standardized.csv`

In [ ]:
from pathlib import Path

import pandas as pd
from pandas.tseries.holiday import (
    AbstractHolidayCalendar,
    GoodFriday,
    Holiday,
    USLaborDay,
    USMartinLutherKingJr,
    USMemorialDay,
    USPresidentsDay,
    USThanksgivingDay,
    nearest_workday,
)
from pandas.tseries.offsets import CustomBusinessDay, DateOffset

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

INPUT_PATH = ROOT / 'data' / 'processed' / '03_candidate_events.csv'
OUTPUT_PATH = ROOT / 'data' / 'processed' / 'standardized' / '03_candidate_events_standardized.csv'
FINBERT_RAW_DIR = ROOT / 'data' / 'processed' / 'un-standardized'
FINBERT_PROCESSED_DIR = ROOT / 'data' / 'processed'

FINBERT_INPUTS = [
    '04_finbert_scored_all.csv',
    '04_finbert_scored_first_term.csv',
    '04_finbert_scored_second_term.csv',
]

REQUIRED_COLUMNS = {
    'event_id',
    'text_id',
    'source',
    'term_id',
    'event_time',
    'event_time_est',
    'market_effective_date',
    'label_window_end_48h',
}

print('Project root:', ROOT)

In [ ]:
class NyseHolidayCalendar(AbstractHolidayCalendar):
    rules = [
        Holiday('NewYearsDay', month=1, day=1, observance=nearest_workday),
        USMartinLutherKingJr,
        USPresidentsDay,
        GoodFriday,
        USMemorialDay,
        Holiday('Juneteenth', month=6, day=19, observance=nearest_workday, start_date='2022-06-19'),
        Holiday('IndependenceDay', month=7, day=4, observance=nearest_workday),
        USLaborDay,
        USThanksgivingDay,
        Holiday('ChristmasDay', month=12, day=25, observance=nearest_workday),
        Holiday('BushFuneral', year=2018, month=12, day=5),
        Holiday('CarterFuneral', year=2025, month=1, day=9),
    ]


def load_finbert_frames() -> list[pd.DataFrame]:
    frames = []
    for name in FINBERT_INPUTS:
        path = FINBERT_RAW_DIR / name
        if not path.exists():
            path = FINBERT_PROCESSED_DIR / name
        frames.append(pd.read_csv(path))
    return frames


def collect_trading_days(frames) -> pd.DatetimeIndex:
    published_values = []
    for df in frames:
        parsed = pd.to_datetime(df['published_at_est'], utc=True, errors='coerce')
        if parsed.notna().any():
            local = parsed.dt.tz_convert('America/New_York').dt.normalize()
            published_values.append(local.dropna())

    if not published_values:
        raise ValueError('Could not infer overall date range from published_at_est.')

    all_dates = pd.concat(published_values)
    start = all_dates.min() - DateOffset(days=10)
    end = all_dates.max() + DateOffset(days=10)

    cbd = CustomBusinessDay(calendar=NyseHolidayCalendar())
    trading_days = pd.date_range(start=start, end=end, freq=cbd, tz='America/New_York')
    return pd.DatetimeIndex(trading_days)


def next_trading_day(trading_days: pd.DatetimeIndex, current_day: pd.Timestamp) -> pd.Timestamp:
    idx = trading_days.searchsorted(current_day, side='right')
    if idx >= len(trading_days):
        raise ValueError(f'No next trading day found after {current_day}.')
    return trading_days[idx]


def prev_trading_day(trading_days: pd.DatetimeIndex, current_day: pd.Timestamp) -> pd.Timestamp:
    idx = trading_days.searchsorted(current_day, side='left') - 1
    if idx < 0:
        raise ValueError(f'No previous trading day found before {current_day}.')
    return trading_days[idx]


def derive_target_trade_date(ts_ny: pd.Timestamp, trading_days: pd.DatetimeIndex) -> pd.Timestamp:
    trade_day = ts_ny.normalize()
    is_trading_day = trade_day in trading_days
    market_close = ts_ny.replace(hour=16, minute=0, second=0, microsecond=0)

    if is_trading_day and ts_ny < market_close:
        return trade_day
    if is_trading_day and ts_ny >= market_close:
        return next_trading_day(trading_days, trade_day)

    future_idx = trading_days.searchsorted(trade_day, side='left')
    if future_idx >= len(trading_days):
        raise ValueError(f'No trading day on or after {trade_day}.')
    return trading_days[future_idx]


def derive_feature_anchor_date(ts_ny: pd.Timestamp, trading_days: pd.DatetimeIndex) -> pd.Timestamp:
    trade_day = ts_ny.normalize()
    is_trading_day = trade_day in trading_days
    market_close = ts_ny.replace(hour=16, minute=0, second=0, microsecond=0)

    if is_trading_day and ts_ny >= market_close:
        return trade_day
    return prev_trading_day(trading_days, trade_day)

In [ ]:
df = pd.read_csv(INPUT_PATH)
missing = REQUIRED_COLUMNS - set(df.columns)
if missing:
    missing_str = ', '.join(sorted(missing))
    raise ValueError(f'03_candidate_events.csv missing required columns: {missing_str}')

trading_days = collect_trading_days(load_finbert_frames())

out = df.copy()
out['event_time_utc'] = pd.to_datetime(out['event_time'], utc=True, errors='coerce', format='mixed')
out['event_time_ny'] = pd.to_datetime(out['event_time_est'], utc=True, errors='coerce', format='mixed').dt.tz_convert('America/New_York')

out['feature_anchor_date'] = out['event_time_ny'].apply(
    lambda ts: derive_feature_anchor_date(ts, trading_days) if pd.notna(ts) else pd.NaT
)
out['target_trade_date'] = out['event_time_ny'].apply(
    lambda ts: derive_target_trade_date(ts, trading_days) if pd.notna(ts) else pd.NaT
)

rename_legacy = {
    'event_time': 'legacy_event_time_utc',
    'event_time_est': 'legacy_event_time_est',
    'market_effective_date': 'legacy_market_effective_date',
}
out = out.rename(columns=rename_legacy)

preferred_order = [
    'event_id', 'text_id', 'source', 'term_id', 'event_time_utc', 'event_time_ny',
    'feature_anchor_date', 'target_trade_date', 'label_window_end_48h',
    'candidate_priority', 'keyword_score', 'theme_hits', 'action_hits', 'object_hits',
    'relevance_terms', 'follow_up_count_all_48h', 'follow_up_count_relevant_48h',
    'context_mode', 'review_flag', 'rule_text', 'source_text', 'url',
    'legacy_event_time_utc', 'legacy_event_time_est', 'legacy_market_effective_date',
]
ordered_existing = [col for col in preferred_order if col in out.columns]
remaining = [col for col in out.columns if col not in ordered_existing]
out = out[ordered_existing + remaining]

for col in [
    'event_time_utc', 'event_time_ny', 'feature_anchor_date', 'target_trade_date',
    'label_window_end_48h', 'legacy_event_time_utc', 'legacy_event_time_est', 'legacy_market_effective_date',
]:
    if col in out.columns:
        out[col] = out[col].astype('string')

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(OUTPUT_PATH, index=False)
print(f'Wrote {OUTPUT_PATH.name}: rows={len(out)}, cols={len(out.columns)}')

## Part 6. FinBERT Output Standardization

标准化 FinBERT 输出表，并补齐后续对齐所需日期字段。

# FinBERT Output Standardization

作用：
- 读取 `data/processed/un-standardized/` 下的 FinBERT 输出
- 标准化时间字段
- 推导 `feature_anchor_date` 与 `target_trade_date`
- 写出 `data/processed/standardized/` 下的标准化结果

In [ ]:
from dataclasses import dataclass
from pathlib import Path

import pandas as pd
from pandas.tseries.holiday import (
    AbstractHolidayCalendar,
    EasterMonday,
    GoodFriday,
    Holiday,
    USLaborDay,
    USMartinLutherKingJr,
    USMemorialDay,
    USPresidentsDay,
    USThanksgivingDay,
    nearest_workday,
)
from pandas.tseries.offsets import CustomBusinessDay, DateOffset

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

PROCESSED_DIR = ROOT / 'data' / 'processed'
RAW_DIR = PROCESSED_DIR / 'un-standardized'
STANDARDIZED_DIR = PROCESSED_DIR / 'standardized'

@dataclass(frozen=True)
class FileSpec:
    input_name: str
    output_name: str

FILE_SPECS = [
    FileSpec('04_finbert_scored_all.csv', '04_finbert_standardized_all.csv'),
    FileSpec('04_finbert_scored_first_term.csv', '04_finbert_standardized_first_term.csv'),
    FileSpec('04_finbert_scored_second_term.csv', '04_finbert_standardized_second_term.csv'),
]

REQUIRED_COLUMNS = {
    'text_id',
    'published_at_utc',
    'published_at_est',
    'term_id',
    'source',
    'finbert_input_text',
    'finbert_label',
    'finbert_pos',
    'finbert_neu',
    'finbert_neg',
}

print('Project root:', ROOT)

In [ ]:
class NyseHolidayCalendar(AbstractHolidayCalendar):
    rules = [
        Holiday('NewYearsDay', month=1, day=1, observance=nearest_workday),
        USMartinLutherKingJr,
        USPresidentsDay,
        GoodFriday,
        USMemorialDay,
        Holiday('Juneteenth', month=6, day=19, observance=nearest_workday, start_date='2022-06-19'),
        Holiday('IndependenceDay', month=7, day=4, observance=nearest_workday),
        USLaborDay,
        USThanksgivingDay,
        Holiday('ChristmasDay', month=12, day=25, observance=nearest_workday),
        Holiday('BushFuneral', year=2018, month=12, day=5),
        Holiday('CarterFuneral', year=2025, month=1, day=9),
    ]


def load_frames() -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for spec in FILE_SPECS:
        path = RAW_DIR / spec.input_name
        if not path.exists():
            path = PROCESSED_DIR / spec.input_name
        df = pd.read_csv(path)
        missing = REQUIRED_COLUMNS - set(df.columns)
        if missing:
            missing_str = ', '.join(sorted(missing))
            raise ValueError(f'{spec.input_name} missing required columns: {missing_str}')
        frames[spec.input_name] = df
    return frames


def collect_trading_days(frames) -> pd.DatetimeIndex:
    published_values = []
    for df in frames:
        parsed = pd.to_datetime(df['published_at_est'], utc=True, errors='coerce')
        if parsed.notna().any():
            local = parsed.dt.tz_convert('America/New_York').dt.normalize()
            published_values.append(local.dropna())

    if not published_values:
        raise ValueError('Could not infer overall date range from published_at_est.')

    all_dates = pd.concat(published_values)
    start = all_dates.min() - DateOffset(days=10)
    end = all_dates.max() + DateOffset(days=10)

    cbd = CustomBusinessDay(calendar=NyseHolidayCalendar())
    trading_days = pd.date_range(start=start, end=end, freq=cbd, tz='America/New_York')
    return pd.DatetimeIndex(trading_days)


def next_trading_day(trading_days: pd.DatetimeIndex, current_day: pd.Timestamp) -> pd.Timestamp:
    idx = trading_days.searchsorted(current_day, side='right')
    if idx >= len(trading_days):
        raise ValueError(f'No next trading day found after {current_day}.')
    return trading_days[idx]


def prev_trading_day(trading_days: pd.DatetimeIndex, current_day: pd.Timestamp) -> pd.Timestamp:
    idx = trading_days.searchsorted(current_day, side='left') - 1
    if idx < 0:
        raise ValueError(f'No previous trading day found before {current_day}.')
    return trading_days[idx]


def derive_target_trade_date(ts_ny: pd.Timestamp, trading_days: pd.DatetimeIndex) -> pd.Timestamp:
    trade_day = ts_ny.normalize()
    is_trading_day = trade_day in trading_days
    market_close = ts_ny.replace(hour=16, minute=0, second=0, microsecond=0)

    if is_trading_day and ts_ny < market_close:
        return trade_day
    if is_trading_day and ts_ny >= market_close:
        return next_trading_day(trading_days, trade_day)

    future_idx = trading_days.searchsorted(trade_day, side='left')
    if future_idx >= len(trading_days):
        raise ValueError(f'No trading day on or after {trade_day}.')
    return trading_days[future_idx]


def derive_feature_anchor_date(ts_ny: pd.Timestamp, trading_days: pd.DatetimeIndex) -> pd.Timestamp:
    trade_day = ts_ny.normalize()
    is_trading_day = trade_day in trading_days
    market_close = ts_ny.replace(hour=16, minute=0, second=0, microsecond=0)

    if is_trading_day and ts_ny >= market_close:
        return trade_day
    return prev_trading_day(trading_days, trade_day)


def standardize_frame(df: pd.DataFrame, trading_days: pd.DatetimeIndex) -> pd.DataFrame:
    out = df.copy()

    out['published_at_utc'] = pd.to_datetime(out['published_at_utc'], utc=True, errors='coerce')
    out['published_at_ny'] = pd.to_datetime(out['published_at_est'], utc=True, errors='coerce', format='mixed').dt.tz_convert('America/New_York')

    out['feature_anchor_date'] = out['published_at_ny'].apply(
        lambda ts: derive_feature_anchor_date(ts, trading_days) if pd.notna(ts) else pd.NaT
    )
    out['target_trade_date'] = out['published_at_ny'].apply(
        lambda ts: derive_target_trade_date(ts, trading_days) if pd.notna(ts) else pd.NaT
    )

    rename_legacy = {
        'published_at_est': 'legacy_published_at_est',
        'market_effective_date': 'legacy_market_effective_date',
        'feature_alignment_date': 'legacy_feature_alignment_date',
        'label_window_end_t2': 'legacy_label_window_end_t2',
    }
    out = out.rename(columns=rename_legacy)

    preferred_order = [
        'text_id', 'source', 'term_id', 'published_at_utc', 'published_at_ny',
        'feature_anchor_date', 'target_trade_date', 'text_raw', 'text_clean',
        'text_finbert', 'finbert_input_text', 'finbert_label', 'finbert_pos',
        'finbert_neu', 'finbert_neg', 'is_trump_direct_text', 'is_retweet',
        'url', 'label_window_end_48h', 'legacy_published_at_est',
        'legacy_market_effective_date', 'legacy_feature_alignment_date',
        'legacy_label_window_end_t2', 'device', 'in_market_hours', 'text_hash',
    ]

    ordered_existing = [col for col in preferred_order if col in out.columns]
    remaining = [col for col in out.columns if col not in ordered_existing]
    return out[ordered_existing + remaining]


def format_for_csv(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for col in [
        'published_at_utc', 'published_at_ny', 'feature_anchor_date', 'target_trade_date',
        'label_window_end_48h', 'legacy_published_at_est', 'legacy_market_effective_date',
        'legacy_feature_alignment_date', 'legacy_label_window_end_t2',
    ]:
        if col in out.columns:
            out[col] = out[col].astype('string')
    return out

In [ ]:
frames = load_frames()
trading_days = collect_trading_days(frames.values())

print(f'Inferred trading days: {len(trading_days)}')
print(f'First trading day: {trading_days.min()}')
print(f'Last trading day: {trading_days.max()}')

STANDARDIZED_DIR.mkdir(parents=True, exist_ok=True)

for spec in FILE_SPECS:
    df = standardize_frame(frames[spec.input_name], trading_days)
    df = format_for_csv(df)
    output_path = STANDARDIZED_DIR / spec.output_name
    df.to_csv(output_path, index=False)
    print(f'Wrote {spec.output_name}: rows={len(df)}, cols={len(df.columns)}')